# Experiment 6: external test on Zambia and Kenya (Notebook 06)

**Question.** Do models trained on all of CODA work on Zambia and Kenya, and do the shortcut and threshold findings hold there?
Zambia and Kenya differ from CODA in devices, settings and patients. Cutting **all three datasets to identical 0.5 s clips** also settles the old concern that Zambia's clips were padded with silence.

**Owner:** Michael (Workstream D). **Status:** draft, written before the first run on real data. Read every printed check on the first run.

## Before you press Run: Kaggle settings

1. **Add data (all private, all small):** (a) the Zambia bundles from Notebook 01: `zambia_segmented_coughs.zip` and `zambia_metadata_cohort.zip` (your dataset `zambia-outputs-michael`); (b) optionally the output of your private run of Notebook 04, which holds `coda_folds.csv` and the cached CODA waveforms. Do **not** attach the 84 GB Zambia source dataset: this notebook searches `/kaggle/input` for its files.
2. **Secret:** `AZURE_STORAGE_CONNECTION_STRING`. The same secret serves the `coda` and `tbscreen` containers if they are in one storage account.
3. **Accelerator:** GPU T4 only for the VGGish embedding step; the models themselves run on CPU. **Internet: ON** (Azure, VGGish weights).
4. Keep the notebook **private**. Exports contain aggregate tables and figures only, never audio, embeddings or per-participant predictions.

## What this notebook does

| Step | What | Output |
|---|---|---|
| 1 | Connect to the `coda` and `tbscreen` containers | |
| 2 | CODA: participants, nine questions, folds, 0.5 s clips | training data |
| 3 | Zambia: cohort, clips cut to 0.5 s, questions | test data |
| 4 | Kenya: passive coughs (one device, up to 10 per person) cut to 0.5 s, questions | test data |
| 5 | Loudness and length check across the three datasets | Table E6a |
| 6 | MFCC, spectrogram and VGGish features, identical for all datasets | |
| 7 | Train on all of CODA, test on Zambia and on Kenya, frozen 90 % sensitivity threshold from CODA out-of-fold predictions | **Table E6** |
| 8 | Zambia to Kenya and Kenya to Zambia with the new clips, against the old results | **Table E6b** |
| 9 | Nine-country probe | **Figure E6** |
| 10 | Optional: train on Zambia and Kenya, test on each CODA group | |
| 11 | Save results in the Appendix B format and package them | `e6_external/` |

## Rules this notebook follows (from the roadmap)

* **Same people, same groups:** CODA folds come from `coda_folds.csv` of Notebook 04 (same seed, never re-split).
* **No peeking:** no label of Zambia or Kenya is used for anything except scoring. Every setting is fixed in advance or chosen on CODA.
* **Same operating point:** the threshold for 90 % sensitivity is fixed on CODA out-of-fold predictions and applied unchanged.
* **Data protection:** nothing derived from audio leaves the session except aggregate tables and figures.

## Step 0. Setup

**What we do.** Import libraries and set every parameter in one place.

**Choices fixed in advance (not tuned on any test result).**

* `KENYA_DEVICE = "pixel"`: TBscreen recorded each cough on a phone and two microphones. We use the **phone**, because CODA's app and the Zambia phones are phones. Change it to `"yeti"` or `"codec"` only as a labelled sensitivity run.
* `KENYA_MAX_PER_PARTICIPANT = 10`: the cap from the plan (about 1,500 files). Coughs are picked by a seeded random draw.
* Zambia is scored **per phone (A, B, C)** and **pooled** (every clip of a person, all phones, averaged).

In [ ]:
import os, io, re, sys, json, time, zipfile, subprocess, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import soundfile as sf, librosa
from scipy.stats import norm
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score
from IPython.display import display
warnings.filterwarnings("ignore")

TEST_MODE = os.environ.get("EXT_TEST_MODE") == "1"
NB_VERSION = "nb06-v1"
CFG = dict(
    CODA_CONTAINER = "coda",      CODA_PREFIX = "v1/Train/",
    TB_CONTAINER   = "tbscreen",  TB_PREFIX   = "v1/",
    LOCAL_ROOT     = None,        # offline testing only: a folder with one sub-folder per container
    # attached inputs (None = search INPUT_ROOT)
    INPUT_ROOT         = "/kaggle/input",
    ZAMBIA_INPUT       = None,    # folder or zip holding the Notebook 01 bundles
    CODA_FOLDS_INPUT   = None,    # folder holding coda_folds.csv from Notebook 04
    CODA_CACHE_INPUT   = None,    # folder holding coda_wave_*.npy / coda_index_*.csv from Notebook 04
    WORK_DIR           = "/kaggle/working",
    # data choices
    CLIP_SECONDS       = 0.5,
    MIN_DETECTOR_SCORE = 0.0,     # CODA: optionally drop coughs the detector was less sure about
    MIN_TB_PER_GROUP   = 15,      # CODA held-out groups (only used if coda_folds.csv is not found)
    KENYA_DEVICE       = "pixel",
    KENYA_MAX_PER_PARTICIPANT = 10,
    ZAMBIA_PHONES      = ["Phone A", "Phone B", "Phone C"],
    # models (identical to Notebook 04)
    SENSITIVITY_TARGET = 0.90,
    N_FOLDS            = 5,
    INNER_K            = 4,
    PROBE_C            = 0.1,
    CLIN_C             = 1.0,
    MAX_MISSINGNESS_GAP = 0.20,   # drop a question if its missingness differs by more than this between TB groups
    DOWNLOAD_WORKERS   = 32,
    N_BOOT             = 1000,
    PROBE_MAX_CLIPS    = 10,      # nine-country probe: at most this many clips per person
    RUN_OPTIONAL_REVERSE = False, # Step 10
    SEED               = 42,
)
if TEST_MODE:
    CFG.update(LOCAL_ROOT="/home/claude/ext_test/fixture_stores", ZAMBIA_INPUT="/home/claude/ext_test/zambia_export",
               INPUT_ROOT="/nonexistent", WORK_DIR="/home/claude/ext_test/work", N_BOOT=100, MIN_TB_PER_GROUP=4,
               KENYA_MAX_PER_PARTICIPANT=4, RUN_OPTIONAL_REVERSE=True)

WORK = Path(CFG["WORK_DIR"]); OUT = WORK / "outputs_external"; CACHE = WORK / "cache_external"
RES = OUT / "results" / "e6_external"; FIG_DIR = OUT / "figures" / "external"
for d in (OUT, CACHE, RES, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CFG["SEED"])
sns.set_theme(style="whitegrid"); plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

try:
    import torch, torch.nn as nn
    TORCH_OK, DEVICE = True, ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(CFG["SEED"])
except ImportError:
    TORCH_OK, DEVICE = False, "cpu"
print(f"Test mode: {TEST_MODE} | PyTorch: {TORCH_OK} | device: {DEVICE}")

## Step 1. Connect to the data

**What we do.** One small wrapper opens an Azure container (or a local folder when testing offline). `fetch_many` downloads many small files in parallel and retries failures. The same wrapper opens `coda` and `tbscreen`.

In [ ]:
class LocalStore:
    # A folder that mirrors the container; blob names are paths relative to it.
    def __init__(self, root):
        self.root = Path(root)
    def walk(self, prefix):
        p = self.root / prefix
        dirs = [f"{prefix}{c.name}/" for c in sorted(p.iterdir()) if c.is_dir()] if p.is_dir() else []
        files = {f"{prefix}{c.name}": c.stat().st_size for c in sorted(p.iterdir()) if c.is_file()} if p.is_dir() else {}
        return dirs, files
    def list(self, prefix):
        p = self.root / prefix
        for f in sorted(p.rglob("*")) if p.is_dir() else []:
            if f.is_file():
                yield f.relative_to(self.root).as_posix(), f.stat().st_size
    def get(self, name):
        return (self.root / name).read_bytes()

class AzureStore:
    def __init__(self, conn, container):
        from azure.storage.blob import BlobServiceClient
        self.c = BlobServiceClient.from_connection_string(conn).get_container_client(container)
    def walk(self, prefix):
        dirs, files = [], {}
        for item in self.c.walk_blobs(name_starts_with=prefix, delimiter="/"):
            if item.name.endswith("/"):
                dirs.append(item.name)
            else:
                files[item.name] = item.size
        return dirs, files
    def list(self, prefix):
        for b in self.c.list_blobs(name_starts_with=prefix):
            if b.size:
                yield b.name, b.size
    def get(self, name):
        return self.c.download_blob(name, max_concurrency=1).readall()

def azure_connection_string():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("AZURE_STORAGE_CONNECTION_STRING")
    except Exception:
        return os.environ["AZURE_STORAGE_CONNECTION_STRING"]

if not CFG["LOCAL_ROOT"]:
    try:
        import azure.storage.blob  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "azure-storage-blob"], check=True)
    _conn = azure_connection_string()

def open_store(container):
    if CFG["LOCAL_ROOT"]:
        return LocalStore(Path(CFG["LOCAL_ROOT"]) / container)
    return AzureStore(_conn, container)

STORE_CODA, STORE_TB = open_store(CFG["CODA_CONTAINER"]), open_store(CFG["TB_CONTAINER"])

def fetch_many(store, names, decode, workers=None, label="files", tries=4):
    # Download many small blobs in parallel; returns {name: decoded or None} and reports failures.
    workers = workers or CFG["DOWNLOAD_WORKERS"]
    out, failed, t0, done = {}, [], time.time(), 0
    def one(n):
        for k in range(tries):
            try:
                return n, decode(store.get(n))
            except Exception as exc:
                if k == tries - 1:
                    return n, exc
                time.sleep(1.5 ** k)
    with ThreadPoolExecutor(workers) as pool:
        futs = [pool.submit(one, n) for n in names]
        step = max(1, len(futs) // 10)
        for f in as_completed(futs):
            n, v = f.result(); done += 1
            if isinstance(v, Exception):
                failed.append((n, repr(v)[:200])); out[n] = None
            else:
                out[n] = v
            if done % step == 0 or done == len(futs):
                print(f"  {label}: {done:,}/{len(futs):,} in {time.time() - t0:.0f} s", flush=True)
    if failed:
        print(f"  {len(failed)} {label} failed, e.g. {failed[:2]}")
    return out

def find_first(name, roots):
    # first file called `name` under any of the given folders (None entries are skipped)
    for r in roots:
        if r and Path(r).exists():
            hits = sorted(Path(r).rglob(name))
            if hits:
                return hits[0]
    return None
print("Connected to:", CFG["CODA_CONTAINER"], "and", CFG["TB_CONTAINER"], "(local test folders)" if CFG["LOCAL_ROOT"] else "")

## Step 1b. Audio front end and modelling helpers

**What we do.** Load, unchanged, the functions of Notebook 04: the VGGish input front end, `to_clip` (the loudest 0.5 s window at 16 kHz, used for **every** dataset), the MFCC and VGGish extractors, DeLong's test, bootstrap intervals, the frozen-threshold rule and the logistic-regression helpers. Using the same code is what makes the three datasets comparable.

In [ ]:
# VGGish input front end, ported from Google's AudioSet mel_features.py / vggish_input.py
# (Apache License 2.0), exactly as used to produce TBscreen's stored features.
import numpy as np
import librosa

VGG_SR, VGG_WIN_S, VGG_HOP_S = 16000, 0.025, 0.010
VGG_MELS, VGG_FMIN, VGG_FMAX, VGG_LOG_OFFSET = 64, 125.0, 7500.0, 0.01
VGG_FRAMES = 96


def _frame(data, window_length, hop_length):
    n = 1 + int(np.floor((data.shape[0] - window_length) / hop_length))
    if n < 1:
        raise ValueError(f"signal too short: {data.shape[0]} samples < window {window_length}")
    shape = (n, window_length) + data.shape[1:]
    strides = (data.strides[0] * hop_length,) + data.strides
    return np.lib.stride_tricks.as_strided(data, shape=shape, strides=strides)


def _periodic_hann(n):
    return 0.5 - 0.5 * np.cos(2 * np.pi / n * np.arange(n))


def _hz_to_mel(f):
    return 1127.0 * np.log(1.0 + f / 700.0)


def _mel_matrix(num_mel_bins, num_spectrogram_bins, sr, lo, hi):
    nyq = sr / 2.0
    bins_mel = _hz_to_mel(np.linspace(0.0, nyq, num_spectrogram_bins))
    edges = np.linspace(_hz_to_mel(lo), _hz_to_mel(hi), num_mel_bins + 2)
    m = np.empty((num_spectrogram_bins, num_mel_bins))
    for i in range(num_mel_bins):
        l, c, u = edges[i:i + 3]
        m[:, i] = np.maximum(0.0, np.minimum((bins_mel - l) / (c - l), (u - bins_mel) / (u - c)))
    m[0, :] = 0.0
    return m


def log_mel(data, sr=VGG_SR):
    win = int(round(sr * VGG_WIN_S)); hop = int(round(sr * VGG_HOP_S))
    fft = 2 ** int(np.ceil(np.log(win) / np.log(2.0)))
    frames = _frame(np.ascontiguousarray(data, dtype=np.float64), win, hop)
    spec = np.abs(np.fft.rfft(frames * _periodic_hann(win), fft))
    mel = spec @ _mel_matrix(VGG_MELS, spec.shape[1], sr, VGG_FMIN, VGG_FMAX)
    return np.log(mel + VGG_LOG_OFFSET)


def vggish_examples(wave, sr):
    """Waveform -> array (n_examples, 96, 64), as vggish_input.waveform_to_examples."""
    wave = np.asarray(wave, dtype=np.float64)
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    if sr != VGG_SR:
        wave = librosa.resample(wave, orig_sr=sr, target_sr=VGG_SR)
    lm = log_mel(wave, VGG_SR)
    return np.ascontiguousarray(_frame(lm, VGG_FRAMES, VGG_FRAMES), dtype=np.float32)


def vggish_example_padded(wave, sr, seconds=1.0):
    """Centre a short clip in `seconds` of zeros (as TBscreen did) and return one (96, 64) example."""
    wave = np.asarray(wave, dtype=np.float64)
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    if sr != VGG_SR:
        wave = librosa.resample(wave, orig_sr=sr, target_sr=VGG_SR)
    n = int(round(VGG_SR * seconds))
    wave = librosa.util.pad_center(wave[:n], size=n, mode="constant") if len(wave) < n else wave[:n]
    return vggish_examples(wave, VGG_SR)[0]

In [ ]:
CLIP_N = int(round(VGG_SR * CFG["CLIP_SECONDS"]))
def to_clip(b):
    w, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=True)
    w = w.mean(axis=1)
    dur = len(w) / sr
    if sr != VGG_SR:
        w = librosa.resample(w, orig_sr=sr, target_sr=VGG_SR)
    if len(w) >= CLIP_N:                             # keep the loudest CLIP_N window (CODA clips are already 0.5 s)
        e = np.convolve(w ** 2, np.ones(CLIP_N), "valid"); s = int(np.argmax(e)); w = w[s:s + CLIP_N]
    else:
        w = np.pad(w, ((CLIP_N - len(w)) // 2, CLIP_N - len(w) - (CLIP_N - len(w)) // 2))
    return w.astype(np.float32), sr, dur

In [ ]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-4, 1 - 1e-4); return np.log(p / (1 - p))

def lr_pipe(C, impute=False):
    steps = ([SimpleImputer(strategy="median")] if impute else []) + [StandardScaler(),
             LogisticRegression(C=C, max_iter=5000, class_weight="balanced")]
    return make_pipeline(*steps)

def probe(Xtr, ytr, Xte):
    return lr_pipe(CFG["PROBE_C"]).fit(Xtr, ytr).predict_proba(Xte)[:, 1]

def part_mean(subjects, p):
    return pd.Series(np.asarray(p, dtype=float)).groupby(np.asarray(subjects)).mean()

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    out = np.empty(N); out[J] = T; return out

def delong(y, p_ref, p_new):
    y = np.asarray(y).astype(int); pos = y == 1; m, n = pos.sum(), (~pos).sum()
    aucs, V10, V01 = [], [], []
    for p in (p_ref, p_new):
        p = np.asarray(p, dtype=float); X, Y = p[pos], p[~pos]
        tx, ty, tz = _midrank(X), _midrank(Y), _midrank(np.r_[X, Y])
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        V10.append((tz[:m] - tx) / n); V01.append(1 - (tz[m:] - ty) / m)
    S = np.cov(np.vstack(V10)) / m + np.cov(np.vstack(V01)) / n
    se = np.sqrt(max(S[0, 0] + S[1, 1] - 2 * S[0, 1], 1e-12)); d = aucs[1] - aucs[0]
    return dict(diff=d, lo=d - 1.96 * se, hi=d + 1.96 * se, p=float(2 * norm.sf(abs(d / se))))

def boot_ci(y, p):
    y, p = np.asarray(y), np.asarray(p); r = np.random.default_rng(CFG["SEED"]); out = []
    for _ in range(CFG["N_BOOT"]):
        i = r.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p[i]))
    return np.percentile(out, [2.5, 97.5])

def threshold_for(y, p, sens):
    # Highest threshold that still catches `sens` of the TB cases in the training predictions.
    pos = np.sort(np.asarray(p)[np.asarray(y) == 1])
    return pos[min(len(pos) - 1, int(np.floor((1 - sens) * len(pos))))] if len(pos) else 0.5

def run_protocol(P, clips, X, folds, clin_cols, inner_k=4):
    # For each test fold: train B (questions), audio, and A (questions + out-of-fold audio score) on the rest.
    # Thresholds for the target sensitivity are set on TRAINING predictions only (frozen), then applied to the test fold.
    rows = []
    for f in sorted(pd.unique(folds)):
        te, tr = P.index[folds == f], P.index[folds != f]
        ctr, cte = clips.participant.isin(tr).values, clips.participant.isin(te).values
        a_te = part_mean(clips.participant.values[cte], probe(X[ctr], clips.label.values[ctr], X[cte])).reindex(te)
        a_tr = pd.Series(np.nan, index=tr)
        k = min(inner_k, int(P.loc[tr, "label"].value_counts().min()))
        for itr, iva in StratifiedKFold(k, shuffle=True, random_state=CFG["SEED"]).split(tr, P.loc[tr, "label"]):
            mt, mv = clips.participant.isin(tr[itr]).values, clips.participant.isin(tr[iva]).values
            a_tr.update(part_mean(clips.participant.values[mv], probe(X[mt], clips.label.values[mt], X[mv])))
        ytr = P.loc[tr, "label"].values
        res = pd.DataFrame({"label": P.loc[te, "label"].values, "fold": f, "audio": a_te.values}, index=te)
        thr = {"audio": threshold_for(ytr, a_tr.values, CFG["SENSITIVITY_TARGET"])}
        for name, cols in (("B", clin_cols), ("A", None)):
            Xtr = P.loc[tr, clin_cols].values if name == "B" else np.c_[P.loc[tr, clin_cols].values, logit(a_tr.values)]
            Xte = P.loc[te, clin_cols].values if name == "B" else np.c_[P.loc[te, clin_cols].values, logit(a_te.values)]
            m = lr_pipe(CFG["CLIN_C"], impute=True)
            oof = cross_val_predict(m, Xtr, ytr, cv=StratifiedKFold(k, shuffle=True, random_state=CFG["SEED"]), method="predict_proba")[:, 1]
            thr[name] = threshold_for(ytr, oof, CFG["SENSITIVITY_TARGET"])
            res[name] = m.fit(Xtr, ytr).predict_proba(Xte)[:, 1]
        for name in ("B", "audio", "A"):
            res[f"{name}_flag"] = (res[name] >= thr[name]).astype(int)
        rows.append(res)
    return pd.concat(rows)

def sens_spec(y, flag):
    y, flag = np.asarray(y), np.asarray(flag)
    return (flag[y == 1].mean() if (y == 1).any() else np.nan), (1 - flag[y == 0].mean() if (y == 0).any() else np.nan)

def summarise(pp, setting):
    rows = []
    for m, lab in (("B", "Questions only (B)"), ("audio", "Cough only"), ("A", "Questions + cough (A)")):
        lo, hi = boot_ci(pp.label, pp[m]); se, sp = sens_spec(pp.label, pp[f"{m}_flag"])
        rows.append({"setting": setting, "model": lab, "AUROC": round(roc_auc_score(pp.label, pp[m]), 3),
                     "95% CI": f"{lo:.2f} to {hi:.2f}", "sensitivity at frozen threshold": round(se, 2),
                     "specificity at frozen threshold": round(sp, 2)})
    d = delong(pp.label, pp.B, pp.A)
    rows.append({"setting": setting, "model": "Gain Δ = A − B", "AUROC": round(d["diff"], 3),
                 "95% CI": f"{d['lo']:+.2f} to {d['hi']:+.2f}", "sensitivity at frozen threshold": f"p = {d['p']:.2f}",
                 "specificity at frozen threshold": ""})
    return pd.DataFrame(rows)

In [ ]:
def mfcc_summary(W):
    out = []
    for w in W:
        m = librosa.feature.mfcc(y=w, sr=VGG_SR, n_mfcc=20, n_fft=400, hop_length=160)
        out.append(np.r_[m.mean(axis=1), m.std(axis=1)])
    return np.asarray(out, dtype=np.float32)



VGGISH_URL = "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth"
if TORCH_OK:
    class VGGishBackbone(nn.Module):
        def __init__(self):
            super().__init__()
            def conv(cin, cout):
                return [nn.Conv2d(cin, cout, 3, 1, 1), nn.ReLU(inplace=True)]
            self.features = nn.Sequential(*conv(1, 64), nn.MaxPool2d(2, 2), *conv(64, 128), nn.MaxPool2d(2, 2),
                                          *conv(128, 256), *conv(256, 256), nn.MaxPool2d(2, 2),
                                          *conv(256, 512), *conv(512, 512), nn.MaxPool2d(2, 2))
            self.embeddings = nn.Sequential(nn.Linear(512 * 24, 4096), nn.ReLU(True), nn.Linear(4096, 4096), nn.ReLU(True),
                                            nn.Linear(4096, 128), nn.ReLU(True))
        def forward(self, x):
            x = self.features(x).permute(0, 2, 3, 1).contiguous()
            return self.embeddings(x.view(x.size(0), -1))
    def vggish_embed(X, batch=256):
        wpath = WORK / "vggish-10086976.pth"
        if not wpath.exists():
            torch.hub.download_url_to_file(VGGISH_URL, str(wpath), hash_prefix="10086976")
        try:
            sd = torch.load(wpath, map_location="cpu", weights_only=True)
        except TypeError:
            sd = torch.load(wpath, map_location="cpu")
        bb = VGGishBackbone(); bb.load_state_dict(sd, strict=True); bb.eval().to(DEVICE)
        out = []
        with torch.no_grad():
            for i in range(0, len(X), batch):
                out.append(bb(torch.as_tensor(X[i:i + batch], device=DEVICE)[:, None]).cpu().numpy())
        return np.concatenate(out).astype(np.float32)

## Step 2. CODA: the training data

**What we do.** Rebuild CODA's participant table with the nine shared questions exactly as in Notebook 04, then load `coda_folds.csv` so that the held-out groups are the same as in every other experiment, and load the 9,772 solicited coughs as 0.5 s clips.

**What to look for.** About 1,082 participants with coughs and 291 with TB, and 9,772 coughs. If `coda_folds.csv` is not found, the notebook rebuilds it with the same code and seed and says so: use that only if Notebook 04's file cannot be attached, and tell the team.

In [ ]:
STORE = STORE_CODA                     # the participant-table code below reads through this name
meta_files = dict(STORE_CODA.list(CFG["CODA_PREFIX"] + "meta_data/"))
def find(pattern):
    hits = sorted(n for n in meta_files if re.search(pattern, Path(n).name, re.I))
    return hits[0] if hits else None
LAYOUT = {"clinical": find(r"clinical_meta_info\.csv$"), "additional": find(r"additional_variables.*\.csv$"),
          "solicited": find(r"solicited_meta_info\.csv$"), "dictionary": find(r"dictionary.*\.xlsx?$"),
          "audio_dir": CFG["CODA_PREFIX"] + "raw_data/solicited_data/"}
display(pd.DataFrame({"what": list(LAYOUT), "found at": list(LAYOUT.values())}))
missing = [k for k in ("clinical", "additional", "solicited") if LAYOUT[k] is None]
assert not missing, f"Not found under {CFG['CODA_PREFIX']}meta_data/: {missing}"

In [ ]:
def read_table(name):
    raw = STORE.get(name)
    if name.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(io.BytesIO(raw))
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue

clin_raw, add_raw, sol = read_table(LAYOUT["clinical"]), read_table(LAYOUT["additional"]), read_table(LAYOUT["solicited"])
if LAYOUT["dictionary"]:
    try:
        print("Data dictionary:"); display(read_table(LAYOUT["dictionary"]))
    except Exception as exc:
        print("Could not read the data dictionary:", exc)

yn = lambda s: s.astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
P = pd.DataFrame({"participant": clin_raw.participant.astype(str).str.strip()})
P["label"] = pd.to_numeric(clin_raw.tb_status, errors="coerce")
P["age"] = pd.to_numeric(clin_raw.age, errors="coerce")
P["male"] = clin_raw.sex.astype(str).str.strip().str.lower().map({"male": 1, "female": 0})
P["smoker"] = yn(clin_raw.smoke_lweek)
P["prev_tb"] = yn(clin_raw.tb_prior)                       # "Not sure" -> missing
P["cough_days"] = pd.to_numeric(clin_raw.reported_cough_dur, errors="coerce")
P["fever"], P["weight_loss"], P["night_sweats"] = yn(clin_raw.fever), yn(clin_raw.weight_loss), yn(clin_raw.night_sweats)
P["hemoptysis"] = yn(clin_raw.hemoptysis)
P["heart_rate"] = pd.to_numeric(clin_raw.heart_rate, errors="coerce")
P["temperature"] = pd.to_numeric(clin_raw.temperature, errors="coerce")
h = pd.to_numeric(clin_raw.height, errors="coerce") / 100
P["bmi"] = pd.to_numeric(clin_raw.weight, errors="coerce") / (h * h)
P = P.set_index("participant")
A = add_raw.assign(participant=add_raw.participant.astype(str).str.strip()).set_index("participant")
P["country"] = A.Country.reindex(P.index).astype(str)
P["hiv_pos"] = A.HIVstatus.reindex(P.index).astype(str).str.strip().str.lower().map({"positive": 1, "negative": 0})
P["xpert_grade"] = A.Xpertcombinedsemiquant.reindex(P.index)
micro = A.Microbiologicreferencestandard.reindex(P.index).astype(str).str.lower().map({"positive": 1, "negative": 0})
agree = (micro == P.label)[micro.notna()].mean()
print(f"Participants: {len(P):,} | TB {int(P.label.sum())} | tb_status agrees with the microbiological reference for {agree:.1%}")

SHARED9 = ["age", "male", "smoker", "hiv_pos", "prev_tb", "cough_days", "fever", "weight_loss", "night_sweats"]
EXTRA = ["hemoptysis", "heart_rate", "temperature", "bmi"]
miss = P.groupby("label")[SHARED9 + EXTRA].apply(lambda g: g.isna().mean()).T
miss.columns = ["missing, non-TB", "missing, TB"]
miss["gap"] = (miss.iloc[:, 1] - miss.iloc[:, 0]).abs()
keep = lambda cols: [c for c in cols if miss.loc[c, "gap"] <= 0.20 and miss.loc[c].iloc[:2].max() < 0.5]
CLIN_SETS = {"shared 9": keep(SHARED9), "extended": keep(SHARED9 + EXTRA)}
print("Share missing by TB status:"); display((miss * 100).round(1))
print("Clinical sets:", {k: v for k, v in CLIN_SETS.items()})

sol["participant"] = sol.participant.astype(str).str.strip()
sol["filename"] = sol.filename.astype(str).str.strip()
sol["blob"] = LAYOUT["audio_dir"] + sol.filename.where(sol.filename.str.lower().str.endswith(".wav"), sol.filename + ".wav")
sol = sol[sol.participant.isin(P.index) & (sol.sound_prediction_score >= CFG["MIN_DETECTOR_SCORE"])].reset_index(drop=True)
P["n_coughs"] = sol.groupby("participant").size().reindex(P.index).fillna(0).astype(int)
print(f"Solicited coughs: {len(sol):,} from {(P.n_coughs > 0).sum():,} participants; "
      f"{(P.n_coughs == 0).sum()} participants have none and are left out of the audio analyses")

In [ ]:
PA = P[P.n_coughs > 0].copy()          # people with audio: every model is compared on exactly these people
folds_file = find_first("coda_folds.csv", [CFG["CODA_FOLDS_INPUT"], CFG["INPUT_ROOT"], OUT])
if folds_file is not None:
    F = pd.read_csv(folds_file, index_col=0); F.index = F.index.astype(str)
    assert set(F.index) == set(PA.index), (
        f"coda_folds.csv lists {len(F)} people but this notebook finds {len(PA)} with coughs: use the file from the same data version")
    PA["group"] = F["group"].reindex(PA.index); PA["rfold"] = F["rfold"].reindex(PA.index)
    print("Folds and held-out groups read from", folds_file)
else:
    print("WARNING: coda_folds.csv not found. Rebuilding the groups and folds with the same code and seed as Notebook 04.")
    tb_by_c = PA.groupby("country").label.sum().sort_values()
    groups, pending = {}, []
    for c, n_tb in tb_by_c.items():
        pending.append(c)
        if PA[PA.country.isin(pending)].label.sum() >= CFG["MIN_TB_PER_GROUP"]:
            groups[" + ".join(pending)] = list(pending); pending = []
    if pending:
        smallest = min(groups, key=lambda g: PA[PA.country.isin(groups[g])].label.sum()) if groups else None
        if smallest:
            new = groups.pop(smallest) + pending; groups[" + ".join(new)] = new
        else:
            groups[" + ".join(pending)] = pending
    PA["group"] = PA.country.map({c: g for g, cs in groups.items() for c in cs})
    PA["rfold"] = -1
    strat = PA.country + "_" + PA.label.astype(int).astype(str)
    for k, (_, te) in enumerate(StratifiedKFold(CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"]).split(PA.index, strat)):
        PA.iloc[te, PA.columns.get_loc("rfold")] = k
PA["label"] = PA.label.astype(int)
display(PA.groupby("group").agg(participants=("label", "size"), TB=("label", "sum")).astype(int))
CODA_COLS = list(CLIN_SETS["shared 9"])
print("CODA questions used for training:", CODA_COLS)

In [ ]:
cwf, cidf = CACHE / f"coda_wave_{len(sol)}.npy", CACHE / f"coda_index_{len(sol)}.csv"
cached_w = find_first(cwf.name, [CFG["CODA_CACHE_INPUT"], CFG["INPUT_ROOT"]])
cached_i = find_first(cidf.name, [CFG["CODA_CACHE_INPUT"], CFG["INPUT_ROOT"]])
if cwf.exists() and cidf.exists():
    cached_w, cached_i = cwf, cidf
if cached_w is not None and cached_i is not None:
    CWAVE, C = np.load(cached_w), pd.read_csv(cached_i)
    print(f"Loaded {len(C):,} cached CODA coughs from {cached_w.parent}")
else:
    got = fetch_many(STORE_CODA, sol.blob.tolist(), to_clip, label="CODA coughs")
    ok = sol.blob.map(lambda n: got.get(n) is not None)
    C = sol[ok].reset_index(drop=True)
    C["orig_sr"] = [got[n][1] for n in C.blob]; C["orig_seconds"] = [round(got[n][2], 3) for n in C.blob]
    CWAVE = np.stack([got[n][0] for n in C.blob])
    np.save(cwf, CWAVE); C.to_csv(cidf, index=False)
assert len(C) == len(CWAVE)
keep = C.participant.astype(str).isin(PA.index).values
C, CWAVE = C[keep].reset_index(drop=True), CWAVE[keep]
C["participant"] = C.participant.astype(str)
C["label"] = C.participant.map(PA.label).astype(int); C["country"] = C.participant.map(PA.country)
print(f"CODA coughs: {CWAVE.shape} from {C.participant.nunique():,} participants")

## Step 3. Zambia: the first test country

**What we do.** Read the Notebook 01 bundles (cohort, harmonised questions, segmented coughs), keep the **analysis cohort** (Chawama and Kanyama, Phones A, B and C, with a TB label), and apply the **same `to_clip`** as for CODA to every segmented cough: the loudest 0.5 s at 16 kHz. This replaces the old one-second padded input.

**What to look for.** About 325 people, 57 with TB, and about 5,287 clips. Small differences from these numbers are expected if you re-ran segmentation; large ones mean something went wrong upstream.

**Question definitions differ across datasets** (a limitation to report): smoking is *smoked in the last week* in CODA but *current or former smoker* in Zambia; cough duration is a day count in CODA and a category midpoint in Zambia. Models trained on CODA see Zambia's version of each answer.

In [ ]:
def find_input(marker, zip_hint, name, roots):
    # Return a folder containing `marker`, extracting a matching zip into WORK/restored/<name> if needed.
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        hits = sorted(p for p in root.rglob(marker) if "restored" not in p.parts)
        if hits:
            return hits[0].parent
        for zp in sorted(root.rglob(f"*{zip_hint}*.zip")):
            with zipfile.ZipFile(zp) as z:
                if any(n == marker or n.endswith("/" + marker) for n in z.namelist()):
                    dest = WORK / "restored" / name
                    if not any(dest.rglob(marker)):
                        print(f"Extracting {zp.name} ...")
                        z.extractall(dest)
                    return next(dest.rglob(marker)).parent
    return None


Z_ROOTS = [CFG["ZAMBIA_INPUT"]] if CFG["ZAMBIA_INPUT"] else [CFG["INPUT_ROOT"]]

def find_seg_dir(roots):
    # A usable folder holds segmentation_manifest.csv AND every clip it lists. A partial copy (for example a stray,
    # half-listed segmented_coughs folder in the dataset) is skipped, and the zip bundle is used instead.
    for root in roots:
        if not (root and Path(root).exists()):
            continue
        for m in sorted(Path(root).rglob("segmentation_manifest.csv")):
            seg = m.parent / "segmented_coughs"
            if "restored" in m.parts or not seg.is_dir():
                continue
            n_rows = sum(1 for _ in open(m)) - 1
            n_wav = sum(1 for _ in seg.rglob("*.wav"))
            if n_wav >= n_rows:
                return m.parent
            print(f"Skipping {m.parent}: only {n_wav:,} of {n_rows:,} clips are there")
    for root in roots:
        if not (root and Path(root).exists()):
            continue
        for zp in sorted(Path(root).rglob("*segmented*.zip")):
            with zipfile.ZipFile(zp) as z:
                if any(n == "segmentation_manifest.csv" or n.endswith("/segmentation_manifest.csv") for n in z.namelist()):
                    dest = WORK / "restored" / "zambia_segmented"
                    if not any(dest.rglob("segmentation_manifest.csv")):
                        print(f"Extracting {zp.name} ...")
                        z.extractall(dest)
                    return next(dest.rglob("segmentation_manifest.csv")).parent
    return None

z_seg_dir = find_seg_dir(Z_ROOTS)
z_meta_dir = find_input("clinical_harmonised.csv", "metadata", "zambia_metadata", Z_ROOTS)
assert z_seg_dir is not None, "A complete zambia_segmented_coughs bundle was not found: attach your Zambia dataset (see the settings above)"
assert z_meta_dir is not None, "zambia_metadata_cohort bundle not found: attach your Zambia dataset (see the settings above)"
Z_SEG_ROOT = z_seg_dir / "segmented_coughs"

z_clin = pd.read_csv(z_meta_dir / "clinical_harmonised.csv").set_index("subject_id")
z_cohort = pd.read_csv(z_meta_dir / "cohort_reproduction.csv").set_index("subject_id")
z_manifest = pd.read_csv(z_seg_dir / "segmentation_manifest.csv")

SHARED9 = ["age", "male", "smoker", "hiv_pos", "prev_tb", "cough_days", "fever", "weight_loss", "night_sweats"]
ZP = z_clin.loc[z_clin.index.intersection(z_cohort.index), ["label", "site"] + SHARED9].copy()
ZP = ZP[ZP.label.notna()]; ZP["label"] = ZP.label.astype(int)
zm = z_manifest[z_manifest.subject_id.isin(ZP.index) & z_manifest.device.isin(CFG["ZAMBIA_PHONES"])]
zm = zm.sort_values(["subject_id", "device", "clip_idx"]).reset_index(drop=True)

absent = [r for r in zm.rel_path if not (Z_SEG_ROOT / r).exists()]
assert not absent, f"{len(absent):,} cohort clips are missing from {Z_SEG_ROOT}, e.g. {absent[:2]}"
def read_clip(rel):
    return to_clip((Z_SEG_ROOT / rel).read_bytes())
t0 = time.time()
with ThreadPoolExecutor(8) as pool:
    zres = list(pool.map(read_clip, zm.rel_path))
print(f"Cut {len(zres):,} Zambia clips in {time.time() - t0:.0f} s")
ZWAVE = np.stack([r[0] for r in zres])
ZC = pd.DataFrame({"participant": zm.subject_id.values, "phone": zm.device.values, "clip_idx": zm.clip_idx.values,
                   "orig_seconds": [round(r[2], 3) for r in zres]})
ZC["label"] = ZC.participant.map(ZP.label).astype(int)
ZP = ZP.loc[ZP.index.isin(ZC.participant)]
print(f"Zambia: {len(ZP)} people ({int(ZP.label.sum())} TB), {len(ZC):,} clips | people per phone:",
      ZC.groupby("phone").participant.nunique().to_dict())
display(ZP.groupby("site").label.agg(people="size", TB="sum"))

## Step 4. Kenya (TBscreen): the second test country

**What we do.** Map the container, read `Metadata.csv`, harmonise the nine questions with Zhang et al.'s published mapping (as in Notebook 03), and list the **passive coughs** of one device (`KENYA_DEVICE`). For every person we keep a seeded random draw of up to `KENYA_MAX_PER_PARTICIPANT` coughs, download them, and apply `to_clip`.

**What to look for.** About 148 people (102 TB, 46 non-TB). **Kenya is case-control** (TB against other respiratory illness), so its absolute AUROC is not comparable with a screening population: read the gain Δ and the threshold transfer. In Kenya, HIV status is missing for most non-TB people; the missingness rule removes any question whose missingness differs by more than 20 points between the two groups, so a model cannot learn "missing means no TB".

In [ ]:
def _norm(s):
    return re.sub(r"[\s_\-]+", "", s.lower())
LEAF_DIRS = {"audiofiles", "scalogramnumpy", "vggishembeddings", "pytorchembeddings"}
def map_layout(store, prefix, max_depth=5):
    found_dirs, small_files, queue = [], {}, [(prefix, 0)]
    while queue:
        p, d = queue.pop(0)
        dirs, files = store.walk(p)
        small_files.update(files)
        for sd in dirs:
            found_dirs.append(sd)
            if _norm(Path(sd.rstrip("/")).name) not in LEAF_DIRS and d < max_depth:
                queue.append((sd, d + 1))
    return found_dirs, small_files

ALL_DIRS, SMALL_FILES = map_layout(STORE_TB, CFG["TB_PREFIX"])
def find_file(pred):
    hits = sorted(n for n in SMALL_FILES if pred(_norm(Path(n).name)))
    return hits[0] if hits else None
def find_dir(kind, leaf):
    hits = sorted(d for d in ALL_DIRS if _norm(Path(d.rstrip("/")).name) == leaf and kind in _norm(d))
    return hits[0] if hits else None
TB_LAYOUT = {"metadata": find_file(lambda n: n == "metadata.csv"),
             "passive_csv": find_file(lambda n: "passive" in n and n.endswith(".csv")),
             "passive_audio": find_dir("passive", "audiofiles")}
display(pd.DataFrame({"what": list(TB_LAYOUT), "found at": list(TB_LAYOUT.values())}))
assert all(TB_LAYOUT.values()), f"Not found in the tbscreen container: {[k for k, v in TB_LAYOUT.items() if not v]}"

def read_csv_blob(name):
    raw = STORE_TB.get(name)
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):        # files saved from Excel are often not UTF-8
        try:
            return pd.read_csv(io.BytesIO(raw), low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"could not decode {name}")

meta_raw = read_csv_blob(TB_LAYOUT["metadata"])
TB_CLIN_COLS = {"age": "age", "male": "gndr_a", "smoker": "smkr_a", "hiv_pos": "hiv_infected_a", "prev_tb": "prev_tb",
                "cough_days": "cough_drtn", "fever": "fever", "weight_loss": "un_wght_loss", "night_sweats": "nght_swts"}
print("Clinical columns missing from Metadata.csv:", [c for c in TB_CLIN_COLS.values() if c not in meta_raw.columns] or "none")
num = lambda c: pd.to_numeric(meta_raw.get(c), errors="coerce")
subj_col = "subject" if "subject" in meta_raw.columns else meta_raw.columns[0]
tb_clin = pd.DataFrame({"subject_id": meta_raw[subj_col].astype(str).str.strip()})
tb_clin["label"] = meta_raw["Label"].astype(str).str.strip().str.upper().map({"TB": 1, "NTB": 0, "NON-TB": 0})
tb_clin["age"] = num("age")
tb_clin["male"] = 1 - num("gndr_a")
tb_clin["smoker"] = num("smkr_a").map({0: 0, 1: 1, 2: 1})
tb_clin["hiv_pos"] = num("hiv_infected_a").map({0: 0, 1: 1})
tb_clin["prev_tb"] = num("prev_tb").map({0: 0, 1: 1, 2: 1})
tb_clin["cough_days"] = num("cough_drtn") * 7
tb_clin["fever"] = num("fever").map({0: 0, 1: 1})
tb_clin["weight_loss"] = num("un_wght_loss").map({0: 0, 1: 1})
tb_clin["night_sweats"] = num("nght_swts").map({0: 0, 1: 1})
tb_clin = tb_clin.drop_duplicates("subject_id").set_index("subject_id")
print(f"Kenya participants with a TB label: {tb_clin.label.notna().sum()} "
      f"(TB {int((tb_clin.label == 1).sum())}, non-TB {int((tb_clin.label == 0).sum())})")

DEVICES_TB = ("codec", "pixel", "yeti")
def cough_key(path, subject):
    rest = path[len(subject):] if path.startswith(subject) else path
    return subject + "|" + "_".join(t for t in rest.split("_") if t and t.lower() not in DEVICES_TB)
t = read_csv_blob(TB_LAYOUT["passive_csv"])
t["path"] = t["path"].astype(str).str.replace(r".*[\\/]", "", regex=True).str.replace(r"\.wav$", "", regex=True)
if "subject" not in t.columns:
    t["subject"] = t.path.str.extract(r"^(PID_\d+[A-Za-z]?)")[0]
if "device" not in t.columns:
    t["device"] = t.path.str.extract(r"(codec|pixel|yeti)", flags=re.I)[0]
t["subject_id"] = t.subject.astype(str).str.strip(); t["device"] = t.device.astype(str).str.lower()
t["has_audio"] = t["Permission_sound"].astype(str).str.lower().eq("yes") if "Permission_sound" in t.columns else True
t["label"] = t.subject_id.map(tb_clin.label)
print(f"Passive cough files: {len(t):,} | per device: {t.device.value_counts().to_dict()} | without a label: {t.label.isna().sum():,}")
kc = t[(t.device == CFG["KENYA_DEVICE"]) & t.has_audio & t.label.notna()].copy()
kc["label"] = kc.label.astype(int)
kc = kc.sort_values(["subject_id", "path"]).reset_index(drop=True)
kc["_draw"] = np.random.default_rng(CFG["SEED"]).random(len(kc))          # seeded random draw within each person
kc = kc.sort_values(["subject_id", "_draw"]).groupby("subject_id").head(CFG["KENYA_MAX_PER_PARTICIPANT"])
kc = kc.drop(columns="_draw").sort_values(["subject_id", "path"])
kc["blob"] = TB_LAYOUT["passive_audio"] + kc.path + ".wav"
print(f"Kenya ({CFG['KENYA_DEVICE']}): {kc.subject_id.nunique()} people, {len(kc):,} coughs to download")
got = fetch_many(STORE_TB, kc.blob.tolist(), to_clip, label="Kenya coughs")
ok = kc.blob.map(lambda n: got.get(n) is not None).values
kc = kc[ok].reset_index(drop=True)
KWAVE = np.stack([got[n][0] for n in kc.blob])
KC = pd.DataFrame({"participant": kc.subject_id.values, "label": kc.label.values, "path": kc.path.values,
                   "orig_seconds": [round(got[n][2], 3) for n in kc.blob], "orig_sr": [got[n][1] for n in kc.blob]})
KP = tb_clin.loc[tb_clin.index.isin(KC.participant), ["label"] + SHARED9].copy(); KP["label"] = KP.label.astype(int)
print(f"Kenya: {len(KP)} people ({int(KP.label.sum())} TB), {len(KC):,} clips")

## Step 5. Loudness and length check (Table E6a)

**What we do.** Compare the 0.5 s clips of the three datasets on three simple numbers: average level (RMS, dB relative to full scale), peak level, and the share of the clip that is not near-silent (frames within 30 dB of the loudest frame). We also report the **original length** of each cough before it was cut.

**What to look for.** Systematic differences between datasets here are part of what a model can learn instead of TB. Zambia's clips were up to one second long before the cut, CODA's are 0.5 s, and Kenya's vary: the identical 0.5 s window removes length as a clue, but not level.

In [ ]:
def clip_stats(W):
    rms = np.sqrt((W ** 2).mean(axis=1)) + 1e-9
    peak = np.abs(W).max(axis=1) + 1e-9
    n = W.shape[1] // 160 * 160
    e = 10 * np.log10((W[:, :n].reshape(len(W), -1, 160) ** 2).mean(axis=2) + 1e-12)
    return pd.DataFrame({"rms_dbfs": 20 * np.log10(rms), "peak_dbfs": 20 * np.log10(peak),
                         "active_share": (e >= e.max(axis=1, keepdims=True) - 30).mean(axis=1)})

LOUD = pd.concat([
    clip_stats(CWAVE).assign(dataset="CODA", group=C.country.values, orig_seconds=C.orig_seconds.values),
    clip_stats(ZWAVE).assign(dataset="Zambia", group=ZC.phone.values, orig_seconds=ZC.orig_seconds.values),
    clip_stats(KWAVE).assign(dataset="Kenya", group=CFG["KENYA_DEVICE"], orig_seconds=KC.orig_seconds.values)], ignore_index=True)
tab_loud = LOUD.groupby("dataset").agg(clips=("rms_dbfs", "size"),
    rms_median=("rms_dbfs", "median"), rms_q25=("rms_dbfs", lambda s: s.quantile(.25)), rms_q75=("rms_dbfs", lambda s: s.quantile(.75)),
    peak_median=("peak_dbfs", "median"), active_share_median=("active_share", "median"),
    orig_seconds_median=("orig_seconds", "median"), orig_seconds_q75=("orig_seconds", lambda s: s.quantile(.75))).round(2)
display(tab_loud)
tab_loud_g = LOUD.groupby(["dataset", "group"]).agg(clips=("rms_dbfs", "size"), rms_median=("rms_dbfs", "median"),
                                                    active_share_median=("active_share", "median")).round(2)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.boxplot(data=LOUD, x="dataset", y="rms_dbfs", ax=axes[0], showfliers=False, palette="Set2")
axes[0].set(title="Clip level (RMS) by dataset", ylabel="dB relative to full scale", xlabel="")
sns.boxplot(data=LOUD, x="dataset", y="active_share", ax=axes[1], showfliers=False, palette="Set2")
axes[1].set(title="Share of the 0.5 s clip that is not near-silent", ylabel="share", xlabel="")
plt.tight_layout(); save_fig(fig, "e6a_loudness"); plt.show()
tab_loud.to_csv(RES / "e6a_loudness_by_dataset.csv"); tab_loud_g.to_csv(RES / "e6a_loudness_by_group.csv")

## Step 6. The same three representations for every clip

**What we do.** MFCC summary (40 numbers), spectrogram summary (128) and pretrained VGGish (128), computed by the Notebook 04 code for CODA, Zambia and Kenya clips. VGGish needs a GPU only for this step.

**HeAR and OPERA (Week 3).** Add an entry to `EXTRA_ENCODERS`: a function that turns an array of 0.5 s waveforms (`n x 8000`, 16 kHz) into one row per clip, using Odunayo's code from Notebook 05. Every later step picks the new representation up automatically. Keep the **same preparation of the 0.5 s clip** as in Notebook 05 (it is decided there, on CODA's random split).

In [ ]:
EXTRA_ENCODERS = {}      # e.g. {"HeAR (frozen)": hear_embed}   (Week 3)

def featurize(W, name):
    X_IN = np.stack([vggish_example_padded(w, VGG_SR, seconds=1.0) for w in W]).astype(np.float32)
    F = {"MFCC summary": mfcc_summary(W),
         "Spectrogram summary": np.concatenate([X_IN.mean(axis=1), X_IN.std(axis=1)], axis=1)}
    if TORCH_OK:
        ef = CACHE / f"{name}_vggish_{len(W)}.npy"
        F["VGGish (pretrained)"] = np.load(ef) if ef.exists() else vggish_embed(X_IN)
        np.save(ef, F["VGGish (pretrained)"])
    for rep, fn in EXTRA_ENCODERS.items():
        ef = CACHE / f"{name}_{re.sub(r'[^a-z0-9]+', '_', rep.lower())}_{len(W)}.npy"
        F[rep] = np.load(ef) if ef.exists() else np.asarray(fn(W), dtype=np.float32)
        np.save(ef, F[rep])
    return F

FC, FZ, FK = featurize(CWAVE, "coda"), featurize(ZWAVE, "zambia"), featurize(KWAVE, "kenya")
REPS = list(FC)
assert REPS == list(FZ) == list(FK)
print("Representations:", {k: v.shape[1] for k, v in FC.items()})

## Step 7. Train on all of CODA, test on Zambia and on Kenya (Table E6)

**What we do.** Three models, exactly as in Notebook 04, trained on **all** CODA participants:

* **B, questions only:** regularised logistic regression on the nine shared questions.
* **Cough only:** a linear probe per clip; a person's score is the mean over their clips.
* **A, questions + cough:** B's inputs plus the cough score. The cough score used in training is an **out-of-fold** score (inner folds), so it is always a prediction for people the cough model did not see.

The **threshold for 90 % sensitivity** is fixed on CODA's out-of-fold predictions and applied unchanged to Zambia and to Kenya, as a clinic would have to do. Nothing from the test country (scaling, imputation, threshold, labels) is used in training.

**Questions that cannot be used.** A question is dropped, for both training and testing, if in the *test* dataset its missingness differs by more than 20 points between TB groups (or exceeds 50 %). Kenya's HIV status is the expected case. The kept questions are printed for every test set.

**Reported:** AUROC for B, cough and A with bootstrap intervals; the gain Δ = A − B with DeLong's 95 % interval on the same people; sensitivity and specificity at the frozen threshold. Each row goes into a CSV in the Appendix B format.

In [ ]:
def usable_cols(Pt, cols):
    # the same missingness rule as Notebook 04: a question whose missingness depends on TB status is a shortcut
    miss = Pt.groupby("label")[cols].apply(lambda g: g.isna().mean()).T
    gap = (miss.iloc[:, 1] - miss.iloc[:, 0]).abs()
    return [c for c in cols if gap[c] <= CFG["MAX_MISSINGNESS_GAP"] and miss.loc[c].max() < 0.5]

def sens_spec(y, flag):
    y, flag = np.asarray(y), np.asarray(flag)
    return (flag[y == 1].mean() if (y == 1).any() else np.nan), (1 - flag[y == 0].mean() if (y == 0).any() else np.nan)

def fit_external(Ptr, part_tr, X_tr, Pte, part_te, X_te, clin_cols):
    # Train B, audio and A on (Ptr, clips of Ptr) and score Pte. Thresholds come from out-of-fold predictions on the training people.
    part_tr, part_te = np.asarray(part_tr), np.asarray(part_te)
    tr = Ptr.index; te = Pte.index
    ytr = Ptr.label.values.astype(int); ylab_tr = Ptr.label.reindex(part_tr).values.astype(int)
    k = min(CFG["INNER_K"], int(pd.Series(ytr).value_counts().min()))
    a_tr = pd.Series(np.nan, index=tr)
    for itr, iva in StratifiedKFold(k, shuffle=True, random_state=CFG["SEED"]).split(tr, ytr):
        mt, mv = np.isin(part_tr, tr[itr]), np.isin(part_tr, tr[iva])
        a_tr.update(part_mean(part_tr[mv], probe(X_tr[mt], ylab_tr[mt], X_tr[mv])))
    a_te = part_mean(part_te, probe(X_tr, ylab_tr, X_te)).reindex(te)
    res = pd.DataFrame({"label": Pte.label.values.astype(int), "audio": a_te.values}, index=te)
    thr = {"audio": threshold_for(ytr, a_tr.values, CFG["SENSITIVITY_TARGET"])}
    for name in ("B", "A"):
        Xtr = Ptr[clin_cols].values if name == "B" else np.c_[Ptr[clin_cols].values, logit(a_tr.values)]
        Xte = Pte[clin_cols].values if name == "B" else np.c_[Pte[clin_cols].values, logit(a_te.values)]
        m = lr_pipe(CFG["CLIN_C"], impute=True)
        oof = cross_val_predict(m, Xtr, ytr, cv=StratifiedKFold(k, shuffle=True, random_state=CFG["SEED"]), method="predict_proba")[:, 1]
        thr[name] = threshold_for(ytr, oof, CFG["SENSITIVITY_TARGET"])
        res[name] = m.fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    for name in ("B", "audio", "A"):
        res[f"{name}_flag"] = (res[name] >= thr[name]).astype(int)
    return res

def b_rows(res, experiment, variant, representation, protocol, test_group, extra=None):
    # one row per model, in the shared results format of Appendix B
    d = delong(res.label, res.B, res.A); rows = []
    for m in ("B", "audio", "A"):
        lo, hi = boot_ci(res.label, res[m]); se, sp = sens_spec(res.label, res[f"{m}_flag"])
        row = dict(experiment=experiment, variant=variant, representation=representation, protocol=protocol,
                   test_group=test_group, model=m, n_people=len(res), n_tb=int(res.label.sum()),
                   auroc=roc_auc_score(res.label, res[m]), auroc_lo=lo, auroc_hi=hi,
                   delta=np.nan, delta_lo=np.nan, delta_hi=np.nan, p=np.nan, sens_frozen=se, spec_frozen=sp,
                   seed=CFG["SEED"], notebook_version=NB_VERSION)
        if m in ("B", "A"):
            row.update(delta=d["diff"], delta_lo=d["lo"], delta_hi=d["hi"], p=d["p"])
        rows.append({**row, **(extra or {})})
    return rows

def external_test(name, Ptr, part_tr, FX_tr, Pte, part_te, FX_te, base_cols, experiment, variant, protocol):
    # Run every representation for one (train, test) pair.
    part_te = np.asarray(part_te); Pte = Pte.loc[Pte.index.isin(part_te)]
    cols = [c for c in base_cols if c in usable_cols(Pte, SHARED9)]
    dropped = [c for c in base_cols if c not in cols]
    print(f"{name}: {len(Pte)} people ({int(Pte.label.sum())} TB) | questions used: {cols} | left out: {dropped or 'none'}")
    keep = np.isin(part_te, Pte.index); rows, preds = [], {}
    for rep in REPS:
        res = fit_external(Ptr, part_tr, FX_tr[rep], Pte, part_te[keep], FX_te[rep][keep], cols)
        preds[rep] = res
        rows += b_rows(res, experiment, variant, rep, protocol, name, extra=dict(questions_used=",".join(cols)))
    return rows, preds

ROWS, PRED = [], {}

In [ ]:
# ---- E6: CODA -> Zambia (pooled over phones, then each phone) and CODA -> Kenya
P_coda = PA[SHARED9 + ["label"]]
part_c = C.participant.values
Z_P_all = ZP[SHARED9 + ["label"]]
settings = [("Zambia, all phones pooled", Z_P_all, ZC.participant.values, FZ, np.ones(len(ZC), bool))]
for ph in CFG["ZAMBIA_PHONES"]:
    settings.append((f"Zambia, {ph}", Z_P_all, ZC.participant.values, FZ, (ZC.phone == ph).values))
for name, Pz, part_z, FZs, mask in settings:
    Pt = Pz.loc[Pz.index.isin(part_z[mask])]
    rows, preds = external_test(name, P_coda, part_c, FC, Pt, part_z[mask], {r: FZs[r][mask] for r in REPS},
                                CODA_COLS, "e6", "coda_to_external", "external")
    ROWS += rows; PRED[name] = preds
rows, preds = external_test(f"Kenya, {CFG['KENYA_DEVICE']}", P_coda, part_c, FC, KP, KC.participant.values, FK,
                            CODA_COLS, "e6", "coda_to_external", "external")
ROWS += rows; PRED[f"Kenya, {CFG['KENYA_DEVICE']}"] = preds

E6 = pd.DataFrame(ROWS)
def readable(df):
    # one line per (test set, representation): AUROC of B, cough, A, gain with interval, and A at the frozen threshold
    out = []
    for (g, rep), d in df.groupby(["test_group", "representation"], sort=False):
        b, a, c = d[d.model == "B"].iloc[0], d[d.model == "A"].iloc[0], d[d.model == "audio"].iloc[0]
        out.append({"test set": g, "representation": rep, "people (TB)": f"{int(a.n_people)} ({int(a.n_tb)})",
                    "AUROC questions (B)": round(b.auroc, 3), "AUROC cough": round(c.auroc, 3), "AUROC both (A)": round(a.auroc, 3),
                    "gain Δ": f"{a.delta:+.3f} ({a.delta_lo:+.2f} to {a.delta_hi:+.2f})", "p": round(a.p, 3),
                    "sens. (A, frozen)": round(a.sens_frozen, 2), "spec. (A, frozen)": round(a.spec_frozen, 2)})
    return pd.DataFrame(out)
TABLE_E6 = readable(E6)
display(TABLE_E6)

## Step 8. Zambia to Kenya and Kenya to Zambia with the new clips (Table E6b)

**What we do.** Repeat the old cross-dataset test with the identical 0.5 s clips, and set the new numbers beside the old ones, to measure **how much clip preparation mattered**. The old numbers are taken from the reference table of the roadmap (questions 0.660 and 0.683, cough 0.551 and 0.585, both 0.644 and 0.679); they were produced in Notebook 03 with the previous clip handling.

**What to look for.** If the cough AUROC moves by much more than its interval, part of the earlier transfer result was a clip-preparation effect. Training uses a different country each time, so each direction has its own question set and threshold.

In [ ]:
Z_all = Z_P_all
E6B = []
for name, Ptr_, part_tr_, FX_tr_, Pte_, part_te_, FX_te_ in (
        ("Zambia to Kenya", Z_all, ZC.participant.values, FZ, KP, KC.participant.values, FK),
        ("Kenya to Zambia", KP, KC.participant.values, FK, Z_all, ZC.participant.values, FZ)):
    cols = [c for c in SHARED9 if c in usable_cols(Ptr_, SHARED9)]          # questions usable in the TRAINING dataset
    rows, preds = external_test(name, Ptr_, part_tr_, FX_tr_, Pte_, part_te_, FX_te_, cols, "e6b", "new_clips", "external")
    E6B += rows; PRED[name] = preds
E6B = pd.DataFrame(E6B)
OLD = {"Zambia to Kenya": dict(B=0.660, audio=0.551, A=0.644), "Kenya to Zambia": dict(B=0.683, audio=0.585, A=0.679)}
old_rows = [dict(experiment="e6b", variant="old_clips_from_notebook_03", representation="as in Notebook 03 (confirm)", protocol="external",
                 test_group=g, model=m, auroc=v, notebook_version="nb03") for g, d in OLD.items() for m, v in d.items()]
E6B_ALL = pd.concat([E6B, pd.DataFrame(old_rows)], ignore_index=True)
cmp_rows = []
for g in OLD:
    for rep in REPS:
        d = E6B[(E6B.test_group == g) & (E6B.representation == rep)]
        cmp_rows.append({"direction": g, "representation": rep,
                         **{f"{m} new": round(d[d.model == m].auroc.iloc[0], 3) for m in ("B", "audio", "A")},
                         **{f"{m} old": OLD[g][m] for m in ("B", "audio", "A")},
                         "gain Δ new": round(d[d.model == "A"].delta.iloc[0], 3)})
TABLE_E6B = pd.DataFrame(cmp_rows)
display(TABLE_E6B)

## Step 9. Nine-country probe (Figure E6)

**What we do.** A simple classifier is asked to name the **country** from a single 0.5 s cough, always tested on people it never saw (five folds, a person's coughs never split across folds). The nine classes are CODA's seven countries (not merged) plus Zambia and Kenya. At most `PROBE_MAX_CLIPS` clips per person are used, so no dataset dominates. A person's prediction is the average over their clips.

**What to look for.** CODA's seven countries used one app everywhere, yet countries were separable at 0.90 AUROC earlier. Zambia and Kenya differ in devices and protocol, so they should be recognised even more easily. A high score means the recording carries a country fingerprint, which a TB model can use as a shortcut. **Note:** Zambia and Kenya are each one country with one study design, so their separability mixes country, device and study design.

In [ ]:
def build_probe_set():
    parts = [("coda", C.participant.values, C.country.values), ("zambia", ZC.participant.values, np.array(["Zambia"] * len(ZC))),
             ("kenya", KC.participant.values, np.array(["Kenya"] * len(KC)))]
    pid = np.concatenate([[f"{n}:{p}" for p in pp] for n, pp, _ in parts]); cty = np.concatenate([cc for _, _, cc in parts])
    pick = []
    r = np.random.default_rng(CFG["SEED"])
    df = pd.DataFrame({"pid": pid, "idx": np.arange(len(pid))})
    for _, g in df.groupby("pid"):
        pick += list(g.idx.values if len(g) <= CFG["PROBE_MAX_CLIPS"] else r.choice(g.idx.values, CFG["PROBE_MAX_CLIPS"], replace=False))
    return pid, cty, np.sort(np.array(pick))

PID, CTY, SEL = build_probe_set()
PROBE_CLASSES = sorted(pd.unique(CTY[SEL]))
person_country = pd.Series(CTY[SEL], index=PID[SEL]).groupby(level=0).first()
print(f"Probe set: {len(SEL):,} clips from {person_country.size:,} people in {len(PROBE_CLASSES)} classes")
display(person_country.value_counts().rename("people").to_frame())

def country_probe(FX_all):
    X, g, y = FX_all[SEL], PID[SEL], CTY[SEL]
    person_pred = {}
    skf = StratifiedGroupKFold(5, shuffle=True, random_state=CFG["SEED"])
    for tr, te in skf.split(X, y, g):
        m = make_pipeline(StandardScaler(), LogisticRegression(C=CFG["PROBE_C"], max_iter=3000, class_weight="balanced"))
        m.fit(X[tr], y[tr])
        pr = pd.DataFrame(m.predict_proba(X[te]), columns=m.classes_).groupby(g[te]).mean()
        for pid_, row in pr.iterrows():
            person_pred[pid_] = row.reindex(PROBE_CLASSES).fillna(0).values
    P_ = pd.DataFrame(person_pred, index=PROBE_CLASSES).T
    truth = person_country.reindex(P_.index)
    yhat = np.array(PROBE_CLASSES)[P_.values.argmax(axis=1)]
    aucs = [roc_auc_score((truth == c).astype(int), P_[c]) for c in PROBE_CLASSES]
    cm = pd.crosstab(truth, pd.Series(yhat, index=P_.index), rownames=["true"], colnames=["predicted"]).reindex(
        index=PROBE_CLASSES, columns=PROBE_CLASSES, fill_value=0)
    return dict(accuracy=accuracy_score(truth, yhat), balanced_accuracy=balanced_accuracy_score(truth, yhat),
                macro_auroc=float(np.mean(aucs)), per_class_auroc=dict(zip(PROBE_CLASSES, np.round(aucs, 3)))), cm

FALL = {rep: np.concatenate([FC[rep], FZ[rep], FK[rep]]) for rep in REPS}
PROBE, CM = {}, {}
for rep in REPS:
    PROBE[rep], CM[rep] = country_probe(FALL[rep])
probe_tab = pd.DataFrame({rep: {"people": int(CM[rep].values.sum()), "accuracy": round(PROBE[rep]["accuracy"], 3),
                                "balanced accuracy": round(PROBE[rep]["balanced_accuracy"], 3),
                                "macro AUROC (one vs rest)": round(PROBE[rep]["macro_auroc"], 3),
                                "chance accuracy": round(1 / len(PROBE_CLASSES), 3)} for rep in REPS}).T
display(probe_tab)
fig, axes = plt.subplots(1, len(REPS), figsize=(6.2 * len(REPS), 5.4))
axes = np.atleast_1d(axes)
for ax, rep in zip(axes, REPS):
    cmn = CM[rep].div(CM[rep].sum(axis=1), axis=0)
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax, annot_kws={"size": 7})
    ax.set(title=f"{rep}: country from one cough\n(row = true country, share predicted)", xlabel="predicted", ylabel="true")
plt.tight_layout(); save_fig(fig, "e6_nine_country_confusion"); plt.show()

## Step 10 (optional). Train on Zambia and Kenya, test on each CODA group

**What we do.** Pool Zambia and Kenya as one training set and test on each held-out CODA group, with the same models, thresholds and rules. This is the reverse direction and is switched on with `RUN_OPTIONAL_REVERSE = True`. It is **exploratory**: the training set is small (about 470 people) and is a mix of a screening and a case-control population.

In [ ]:
E6C = pd.DataFrame()
if CFG["RUN_OPTIONAL_REVERSE"]:
    ZK_P = pd.concat([Z_P_all.assign(pid="z:" + Z_P_all.index.astype(str)), KP.assign(pid="k:" + KP.index.astype(str))]).set_index("pid")
    part_zk = np.concatenate([("z:" + ZC.participant.astype(str)).values, ("k:" + KC.participant.astype(str)).values])
    FZK = {rep: np.concatenate([FZ[rep], FK[rep]]) for rep in REPS}
    cols_zk = [c for c in CODA_COLS if c in usable_cols(ZK_P, SHARED9)]
    rows_c = []
    for g_name, Pg in PA.groupby("group"):
        Pg = Pg[SHARED9 + ["label"]]
        rows, preds = external_test(f"CODA {g_name}", ZK_P[SHARED9 + ["label"]], part_zk, FZK, Pg, C.participant.values, FC,
                                    cols_zk, "e6c", "zambia_kenya_to_coda", "external")
        rows_c += rows
    E6C = pd.DataFrame(rows_c)
    display(readable(E6C))
else:
    print("Skipped (RUN_OPTIONAL_REVERSE = False).")

## Step 11. Save the results

**What we do.** Write every number to `results/e6_external/` in the Appendix B format (`e6_results.csv`), the readable tables, the figures, and a `notes.md` skeleton that follows the reporting guide (configuration, finding, why this setup, intended result, why we think it happened). Package only these aggregate files.

**What to do after the run.** Fill in `notes.md` in your own words, copy `results/e6_external/` and `figures/external/` from the downloaded bundle into the shared results folder, and check that no file in the bundle contains a participant ID before you commit anything to GitHub. Participant-level predictions are **not** saved on purpose.

In [ ]:
APPB = ["experiment", "variant", "representation", "protocol", "test_group", "model", "n_people", "n_tb", "auroc", "auroc_lo",
        "auroc_hi", "delta", "delta_lo", "delta_hi", "p", "sens_frozen", "spec_frozen", "seed", "notebook_version"]
ALL = pd.concat([E6, E6B_ALL] + ([E6C] if len(E6C) else []), ignore_index=True)
ALL[APPB + [c for c in ALL.columns if c not in APPB]].round(4).to_csv(RES / "e6_results.csv", index=False)
TABLE_E6.to_csv(RES / "table_e6_external.csv", index=False)
TABLE_E6B.to_csv(RES / "table_e6b_transfer.csv", index=False)
probe_tab.to_csv(RES / "e6_nine_country_probe.csv")
pd.concat({rep: CM[rep] for rep in REPS}).to_csv(RES / "e6_nine_country_confusion_counts.csv")
pd.DataFrame({rep: PROBE[rep]["per_class_auroc"] for rep in REPS}).to_csv(RES / "e6_nine_country_per_class_auroc.csv")

notes = f"""# Experiment 6: external test on Zambia and Kenya

## Configuration
* Training: all of CODA ({len(PA)} people, {int(PA.label.sum())} TB, {len(C):,} coughs), shared nine questions, folds from coda_folds.csv.
* Test sets: Zambia analysis cohort ({len(ZP)} people, {int(ZP.label.sum())} TB, {len(ZC):,} clips; pooled and per phone); Kenya {CFG['KENYA_DEVICE']} passive coughs ({len(KP)} people, {int(KP.label.sum())} TB, {len(KC):,} clips, at most {CFG['KENYA_MAX_PER_PARTICIPANT']} per person).
* Every clip: loudest 0.5 s at 16 kHz (`to_clip` of Notebook 04). Representations: {', '.join(REPS)}.
* Models, threshold rule and seed as in Notebook 04 (logistic regression, 90 % sensitivity frozen on out-of-fold CODA predictions, seed {CFG['SEED']}).
* Questions left out of a test set when their missingness depends on TB status: see the `questions_used` column of e6_results.csv.
* Notebook version: {NB_VERSION}.

## What did you find?
TODO (use table_e6_external.csv, table_e6b_transfer.csv and the nine-country probe).

## Why did you choose this configuration?
TODO. Fixed before any result: Kenya phone only, 10 coughs per person, Zambia per phone and pooled.

## Were these the intended results?
TODO.

## Why do you think you got these results?
TODO.

## Limitations to state
* Question definitions differ across datasets (smoking, cough duration).
* Kenya is case-control: compare gains and thresholds, not absolute AUROC.
* Small test sets give wide intervals (Zambia has {int(ZP.label.sum())} TB cases).
"""
(RES / "notes.md").write_text(notes)

import zipfile
bundle = WORK / "export" / "e6_external_results.zip"; bundle.parent.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RES.rglob("*")):
        if f.is_file():
            z.write(f, f"results/e6_external/{f.relative_to(RES).as_posix()}")
    for f in sorted(FIG_DIR.rglob("*")):
        if f.is_file():
            z.write(f, f"figures/external/{f.relative_to(FIG_DIR).as_posix()}")
print("Saved", bundle, f"({bundle.stat().st_size / 1e6:.1f} MB). Files:")
print(sorted(f.name for f in RES.iterdir()))
print("Check: no participant IDs in the exports ->",
      not any(str(i) in (RES / 'e6_results.csv').read_text() for i in list(PA.index[:50]) + list(ZP.index[:50]) + list(KP.index[:50])))